# Exercício Extra 01 — KNN do zero (sem scikit-learn)

**Objetivo:** implementar o KNN manualmente (sem usar bibliotecas prontas de ML) para
classificação binária no dataset `emnist_filtrado_AI.npz` (letras **A** e **I** filtradas do EMNIST).

**Regras do exercício:**
- KNN implementado na mão, usando **distância euclidiana**
- Validação cruzada **manual** com **3 folds** para encontrar o melhor valor de K
- Apenas `numpy`/`matplotlib` são usados (nenhuma biblioteca de ML pronta)

**Metodologia adotada:**
1. Carregar o `.npz` e separar um conjunto de teste (holdout) que **nunca** entra na validação cruzada
2. Rodar 3-fold CV no restante dos dados, testando vários valores de K
3. Treinar o KNN final com o melhor K e avaliar no conjunto de teste (holdout)
4. Salvar o "modelo" (dados de treino + K escolhido) para reaproveitar no Exercício Extra 02

## 0. Instalação das dependências

In [1]:
%pip install -q numpy matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Carregamento do dataset

Primeiro inspecionamos as chaves do `.npz` para saber os nomes corretos dos arrays
(varia conforme como o arquivo foi exportado).

In [2]:
import numpy as np

DATA_PATH = "emnist_filtrado_AI.npz"  # ajuste o caminho se o arquivo estiver em outra pasta (ex: "dados/emnist_filtrado_AI.npz")

dados = np.load(DATA_PATH)
print("Chaves disponíveis no arquivo:", list(dados.files))
for chave in dados.files:
    print(f"  {chave}: shape={dados[chave].shape}, dtype={dados[chave].dtype}")

FileNotFoundError: [Errno 2] No such file or directory: 'emnist_filtrado_AI.npz'

Rode a célula acima primeiro pra ver os nomes das chaves impressos, depois ajuste
as duas linhas marcadas abaixo (`X = dados[...]` e `y = dados[...]`) com os nomes corretos.

In [ ]:
# >>> AJUSTE AQUI conforme as chaves impressas na célula anterior <<<
X = dados["X"]   # imagens -- pode vir como (N, H, W) ou já achatado (N, H*W)
y = dados["y"]   # rótulos -- 0 = uma letra (ex: A), 1 = outra (ex: I)

# Garante que X está em formato (N, n_features) -- achatando (flatten) se vier como matriz
if X.ndim == 3:
    IMG_HEIGHT, IMG_WIDTH = X.shape[1], X.shape[2]
    X = X.reshape(X.shape[0], -1)
else:
    # Caso já venha achatado e a imagem original seja quadrada (padrão EMNIST: 28x28)
    IMG_HEIGHT = IMG_WIDTH = int(np.sqrt(X.shape[1]))

X = X.astype(np.float64)
y = np.asarray(y)

print(f"X: {X.shape} | y: {y.shape}")
print(f"Dimensão original da imagem (assumida): {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Classes presentes: {np.unique(y, return_counts=True)}")

## 2. Normalização e separação de um conjunto de teste (holdout)

Normalizamos os pixels para o intervalo `[0, 1]` (ajuda a distância euclidiana a não ser
dominada por escalas grandes) e separamos 20% dos dados como teste final — esse conjunto
fica de fora da validação cruzada, simulando dados "nunca vistos".

In [ ]:
X_norm = X / 255.0 if X.max() > 1.0 else X.copy()

def split_treino_teste(X, y, perc_teste=0.2, seed=42):
    rng = np.random.default_rng(seed)
    n = len(X)
    indices = rng.permutation(n)
    n_teste = int(n * perc_teste)
    idx_teste = indices[:n_teste]
    idx_treino = indices[n_teste:]
    return X[idx_treino], y[idx_treino], X[idx_teste], y[idx_teste]

X_treino_cv, y_treino_cv, X_teste, y_teste = split_treino_teste(X_norm, y, perc_teste=0.2)

print(f"Treino/CV: {X_treino_cv.shape[0]} amostras | Teste (holdout): {X_teste.shape[0]} amostras")

## 3. Implementação do KNN (na mão)

- `distancia_euclidiana`: distância vetorizada entre um ponto de consulta e todas as amostras de treino
- `knn_predict_um`: prediz a classe de **uma** amostra (vota entre os K vizinhos mais próximos)
- `knn_predict`: aplica `knn_predict_um` em um lote de amostras

In [ ]:
def distancia_euclidiana(X_treino, x_consulta):
    # ||a - b|| para cada linha de X_treino contra x_consulta, vetorizado
    return np.sqrt(np.sum((X_treino - x_consulta) ** 2, axis=1))

def knn_predict_um(x_consulta, X_treino, y_treino, k):
    distancias = distancia_euclidiana(X_treino, x_consulta)
    idx_vizinhos = np.argsort(distancias)[:k]
    rotulos_vizinhos = y_treino[idx_vizinhos]
    valores, contagens = np.unique(rotulos_vizinhos, return_counts=True)
    return valores[np.argmax(contagens)]

def knn_predict(X_consulta, X_treino, y_treino, k):
    return np.array([knn_predict_um(x, X_treino, y_treino, k) for x in X_consulta])

## 4. Validação cruzada manual (3 folds) para encontrar o melhor K

Implementação manual do k-fold: embaralha os índices, divide em 3 blocos, e em cada
rodada usa 2 blocos para treino e 1 para validação — sem usar `KFold` do scikit-learn.

In [ ]:
def gerar_folds(n, k_folds=3, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n)
    return np.array_split(indices, k_folds)

def validacao_cruzada_knn(X, y, valores_k, k_folds=3, seed=42):
    folds = gerar_folds(len(X), k_folds, seed)
    acuracias_por_k = {k: [] for k in valores_k}

    for i in range(k_folds):
        idx_val = folds[i]
        idx_treino = np.concatenate([folds[j] for j in range(k_folds) if j != i])
        X_tr, y_tr = X[idx_treino], y[idx_treino]
        X_val, y_val = X[idx_val], y[idx_val]

        for k in valores_k:
            y_pred = knn_predict(X_val, X_tr, y_tr, k)
            acc = np.mean(y_pred == y_val)
            acuracias_por_k[k].append(acc)

    return {k: np.mean(accs) for k, accs in acuracias_por_k.items()}

valores_k = [1, 3, 5, 7, 9, 11, 15]
acc_media_por_k = validacao_cruzada_knn(X_treino_cv, y_treino_cv, valores_k, k_folds=3)

for k, acc in acc_media_por_k.items():
    print(f"K={k:>2} | acurácia média (3-fold) = {acc:.4f}")

melhor_k = max(acc_media_por_k, key=acc_media_por_k.get)
print(f"\nMelhor K encontrado: {melhor_k} (acurácia média = {acc_media_por_k[melhor_k]:.4f})")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(list(acc_media_por_k.keys()), list(acc_media_por_k.values()), marker="o")
plt.axvline(melhor_k, color="red", linestyle="--", label=f"Melhor K = {melhor_k}")
plt.xlabel("K (número de vizinhos)")
plt.ylabel("Acurácia média (3-fold CV)")
plt.title("Validação cruzada — escolha de K")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Avaliação final no conjunto de teste (holdout)

Com o melhor K definido pela validação cruzada, avaliamos no conjunto de teste que
ficou de fora de todo o processo de ajuste — essa é a estimativa mais honesta de
desempenho do modelo.

In [ ]:
y_pred_teste = knn_predict(X_teste, X_treino_cv, y_treino_cv, melhor_k)
acuracia_teste = np.mean(y_pred_teste == y_teste)
print(f"Acurácia no conjunto de teste (K={melhor_k}): {acuracia_teste:.4f}")

def matriz_confusao_manual(y_true, y_pred, classes):
    n = len(classes)
    idx_map = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((n, n), dtype=int)
    for yt, yp in zip(y_true, y_pred):
        cm[idx_map[yt], idx_map[yp]] += 1
    return cm

classes = sorted(np.unique(y).tolist())
cm = matriz_confusao_manual(y_teste, y_pred_teste, classes)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão — KNN (K={melhor_k})")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()
print(cm)

## 6. Salvando o modelo treinado

O "modelo" do KNN são os próprios dados de treino + o valor de K escolhido (não há
parâmetros a ajustar como em outros algoritmos). Salvamos isso em disco para reaproveitar
no **Exercício Extra 02**, sem precisar recarregar o `.npz` original nem repetir a validação cruzada.

In [ ]:
np.savez(
    "modelo_knn_treinado.npz",
    X_treino=X_treino_cv,
    y_treino=y_treino_cv,
    melhor_k=melhor_k,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
)
print("Modelo salvo em 'modelo_knn_treinado.npz' (X_treino, y_treino, melhor_k, dimensões da imagem)")

## 7. Discussão (pontos para a apresentação)

- Como a acurácia variou entre os valores de K testados? O gráfico mostra under/overfitting
  nos extremos (K muito baixo = sensível a ruído; K muito alto = perde detalhe local)?
- A acurácia da validação cruzada (etapa 4) bateu com a acurácia no holdout (etapa 5)?
  Se houver uma diferença grande, o que isso sugere sobre o tamanho/representatividade dos folds?
- Olhando a matriz de confusão: o modelo confunde mais a letra A com I, ou o contrário?
  Faz sentido visualmente (formato das letras)?
- Por que a normalização dos pixels (`/255`) é importante para o KNN especificamente
  (e não necessariamente para todos os algoritmos)?